In [1]:
import spacy
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

print("Checking hardware...")

# --- 1. Detect GPU ---
# Check if CUDA (Nvidia GPU) is available
if torch.cuda.is_available():
    device_id = 0
    device_name = torch.cuda.get_device_name(0)
    print(f"✅ GPU Detected: {device_name}")
else:
    device_id = -1
    print("⚠️ GPU not detected. Running on CPU (Switch 'Accelerator' to GPU in Kaggle settings for speed).")

print("Loading models...")

# --- 2. Load Biomedical NER Model ---
model_name = "d4data/biomedical-ner-all" #alvaroalon2/biobert_diseases_ner, d4data/biomedical-ner-all
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

# Create the pipeline with the detected device
ner_pipeline = pipeline(
    "ner", 
    model=model, 
    tokenizer=tokenizer, 
    aggregation_strategy="max", 
    device=device_id  # <--- This triggers the GPU
)

# --- 3. Load spaCy ---
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("Models loaded successfully.")

2026-02-14 14:02:14.317359: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771077734.522288      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771077734.583687      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771077735.065630      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771077735.065672      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771077735.065675      55 computation_placer.cc:177] computation placer alr

Checking hardware...
✅ GPU Detected: Tesla T4
Loading models...


tokenizer_config.json:   0%|          | 0.00/373 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/266M [00:00<?, ?B/s]

Device set to use cuda:0


Models loaded successfully.


In [2]:
def extract_medical_entities(query):
    """
    Extracts entities using BERT-based NER and lemmatizes them using spaCy.
    """
    # Get NER results
    ner_results = ner_pipeline(query)
    
    entities = []

    for entity in ner_results:
        raw_text = entity["word"]
        
        # Lemmatize the extracted entity text using spaCy
        lemma_doc = nlp(raw_text)
        lemma = " ".join([token.lemma_ for token in lemma_doc])

        entities.append({
            "Original Text": raw_text,
            "Lemmatized": lemma,
            "Entity Group": entity["entity_group"],
            "Confidence": f"{entity['score']:.4f}"
        })

    return entities

In [3]:
# --- Test Cases ---
queries = [
    "Patient reports severe chest pain and shortness of breath.",
    "Prescribed 500mg of Metformin for type 2 diabetes.",
    "MRI revealed a fracture in the distal radius.",
    "I'm a 25 year old female. I have fever,chills and constant coughing. could this indicate a specific illness affecting my respiratory system?"
]

print(f"{'Extraction Results':^60}")
print("="*60)

for q in queries:
    print(f"\nInput: '{q}'")
    results = extract_medical_entities(q)
    
    # Display clearly
    if results:
        for item in results:
            print(f"  - [{item['Entity Group']}] {item['Original Text']} (Lemma: {item['Lemmatized']})")
    else:
        print("  - No entities found.")

                     Extraction Results                     

Input: 'Patient reports severe chest pain and shortness of breath.'
  - [Severity] severe (Lemma: severe)
  - [Biological_structure] chest (Lemma: chest)
  - [Sign_symptom] pain (Lemma: pain)
  - [Sign_symptom] shortness of (Lemma: shortness of)

Input: 'Prescribed 500mg of Metformin for type 2 diabetes.'
  - [Dosage] 500mg (Lemma: 500 mg)
  - [Medication] metformin (Lemma: metformin)

Input: 'MRI revealed a fracture in the distal radius.'
  - [Diagnostic_procedure] mri (Lemma: mri)
  - [Sign_symptom] fracture (Lemma: fracture)

Input: 'I'm a 25 year old female. I have fever,chills and constant coughing. could this indicate a specific illness affecting my respiratory system?'
  - [Age] 25 year old (Lemma: 25 year old)
  - [Sex] female (Lemma: female)
  - [Sign_symptom] chills (Lemma: chill)
